# Morning Checklist Analysis Notebook

This notebook automates the morning checklist analysis based on your actual trading patterns.
Run this each morning to get data-driven insights for your trading day.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

print("Morning Checklist Analysis - Based on Your Actual Trading Data")
print(f"Run Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

# Load your actual trading patterns from the analysis
patterns_df = pd.read_csv('data/trade_patterns.csv', index_col=0)
print("\nLoaded your actual trading patterns from data/trade_patterns.csv")

# Extract patterns for each trade type
YOUR_PATTERNS = {}

# Parse the pattern keys (e.g., "CALL_EXIT", "PUT_RUNNER")
for pattern_key in patterns_df.index:
    parts = pattern_key.split('_')
    if len(parts) >= 2:
        trade_type = parts[0]
        exit_type = '_'.join(parts[1:])
        
        if trade_type not in YOUR_PATTERNS:
            YOUR_PATTERNS[trade_type] = {
                'patterns': {},
                'all_rsi': [],
                'all_durations': [],
                'all_returns': []
            }
        
        YOUR_PATTERNS[trade_type]['patterns'][exit_type] = {
            'count': patterns_df.loc[pattern_key, 'count'],
            'avg_duration': patterns_df.loc[pattern_key, 'avg_duration'],
            'avg_return': patterns_df.loc[pattern_key, 'avg_return'],
            'win_rate': patterns_df.loc[pattern_key, 'profitable_pct'] / 100,
            'rsi_mean': patterns_df.loc[pattern_key, 'entry_rsi14_w_mean'],
            'rsi_std': patterns_df.loc[pattern_key, 'entry_rsi14_w_std']
        }

# Load enriched trades to get comprehensive patterns
enriched_df = pd.read_csv('data/trades_enriched.csv')

# Calculate overall patterns for each trade type
for trade_type in ['CALL', 'PUT']:
    type_data = enriched_df[enriched_df['Trade_Type'] == trade_type]
    if len(type_data) > 0:
        YOUR_PATTERNS[trade_type]['rsi_range'] = (
            type_data['Entry_RSI14_W'].min(),
            type_data['Entry_RSI14_W'].max()
        )
        YOUR_PATTERNS[trade_type]['rsi_avg'] = type_data['Entry_RSI14_W'].mean()
        YOUR_PATTERNS[trade_type]['duration_avg'] = type_data['Duration'].mean()
        YOUR_PATTERNS[trade_type]['avg_return'] = type_data['Return_Pct'].mean()
        YOUR_PATTERNS[trade_type]['win_rate'] = (type_data['Return_Pct'] > 0).mean()
        YOUR_PATTERNS[trade_type]['count'] = len(type_data)
        YOUR_PATTERNS[trade_type]['rvol_avg'] = type_data['Entry_RVOL20'].mean()
        
        # Calculate VWAP position percentages
        below_vwap = (type_data['Entry_Last'] < type_data['Entry_VWAP']).sum()
        total = len(type_data)
        YOUR_PATTERNS[trade_type]['below_vwap_pct'] = below_vwap / total
        YOUR_PATTERNS[trade_type]['above_vwap_pct'] = 1 - YOUR_PATTERNS[trade_type]['below_vwap_pct']

print("\nYour Trading Patterns Summary:")
for trade_type in ['CALL', 'PUT']:
    if trade_type in YOUR_PATTERNS:
        patterns = YOUR_PATTERNS[trade_type]
        print(f"\n{trade_type}:")
        print(f"  RSI Range: {patterns['rsi_range'][0]:.1f} - {patterns['rsi_range'][1]:.1f} (avg: {patterns['rsi_avg']:.1f})")
        print(f"  Win Rate: {patterns['win_rate']*100:.0f}%")
        print(f"  Avg Return: {patterns['avg_return']:.2f}%")
        print(f"  Avg Duration: {patterns['duration_avg']:.0f} min")
        print(f"  Avg RVOL: {patterns['rvol_avg']:.1f}x")
        print(f"  Below VWAP: {patterns['below_vwap_pct']*100:.0f}%")

print("\n" + "="*60)

## 1. Load Latest Market Data

In [ ]:
# Load the most recent IWM data with indicators
import glob

# Find the most recent indicator file
indicator_files = glob.glob('data/historical_iwm_*_with_indicators.csv')
if not indicator_files:
    raise FileNotFoundError("No indicator data found. Please run: python3 iwm_analysis.py")

latest_file = sorted(indicator_files)[-1]
df = pd.read_csv(latest_file)
df['Time'] = pd.to_datetime(df['Time'])

print(f"Loaded {len(df)} records from {latest_file}")

# Get the last trading day's data
latest_date = df['Time'].dt.date.max()
today_data = df[df['Time'].dt.date == latest_date].copy()

# Get last data point
current = today_data.iloc[-1]

print(f"\nLatest Data: {current['Time']}")
print(f"Current Price: ${current['Last']:.2f}")
print(f"Current RSI: {current['RSI14_W']:.1f}")
print(f"Current Volume: {current['Volume']:,.0f}")

## 2. Pre-Market Context Analysis

In [ ]:
# Analyze key levels
print("\n" + "="*60)
print("KEY PRICE LEVELS")
print("="*60)

# Get previous trading day's data
prev_day_date = df[df['Time'].dt.date < latest_date]['Time'].dt.date.max()
yesterday = df[df['Time'].dt.date == prev_day_date]

if len(yesterday) > 0:
    print(f"\nYesterday's Levels ({prev_day_date}):")
    print(f"  High: ${yesterday['High'].max():.2f}")
    print(f"  Low: ${yesterday['Low'].min():.2f}")
    print(f"  Close: ${yesterday.iloc[-1]['Last']:.2f}")

# Today's levels so far
print(f"\nToday's Levels ({latest_date}):")
print(f"  High: ${today_data['High'].max():.2f}")
print(f"  Low: ${today_data['Low'].min():.2f}")
print(f"  Current: ${current['Last']:.2f}")
print(f"  VWAP: ${current['VWAP']:.2f}")

# Price position
above_vwap = current['Last'] > current['VWAP']
print(f"\nPrice vs VWAP: {'ABOVE' if above_vwap else 'BELOW'} VWAP by ${abs(current['Last'] - current['VWAP']):.2f}")

## 3. Trade Pattern Logic Checklist

This section analyzes your trade_patterns.csv to create a comprehensive logical checklist for trade setups.

In [ ]:
# Analyze trade_patterns.csv to create comprehensive logical checklist
print("\n" + "="*60)
print("LOGICAL TRADE SETUP CHECKLIST FROM YOUR PATTERNS")
print("="*60)

# Load the comprehensive patterns file
patterns_full = pd.read_csv('data/trade_patterns.csv', index_col=0)

# Create logic rules for each trade type
for trade_type in ['CALL', 'PUT']:
    print(f"\n{'='*30} {trade_type} TRADE LOGIC {'='*30}")
    
    # Get all patterns for this trade type
    type_patterns = patterns_full[[idx for idx in patterns_full.index if idx.startswith(trade_type)]]
    
    if len(type_patterns) > 0:
        # Find the most profitable exit type
        best_exit = type_patterns.loc[type_patterns['avg_return'].idxmax()]
        best_exit_name = type_patterns.index[type_patterns['avg_return'].argmax()].split('_')[1]
        
        print(f"\n📊 BEST {trade_type} STRATEGY: {best_exit_name} (avg return: {best_exit['avg_return']:.3f}%)")
        print(f"   Win Rate: {best_exit['profitable_pct']:.1f}%")
        print(f"   Avg Duration: {best_exit['avg_duration']:.1f} minutes")
        
        # Create comprehensive checklist based on patterns
        print(f"\n✅ {trade_type} ENTRY CHECKLIST:")
        
        # 1. RSI Logic
        if 'entry_rsi14_w_min' in patterns_full.columns:
            rsi_min = type_patterns['entry_rsi14_w_min'].min()
            rsi_max = type_patterns['entry_rsi14_w_max'].max()
            rsi_25pct = type_patterns['entry_rsi14_w_25pct'].min()
            rsi_75pct = type_patterns['entry_rsi14_w_75pct'].max()
            
            print(f"\n1️⃣ RSI CONDITIONS:")
            print(f"   ✓ RSI must be between {rsi_min:.1f} and {rsi_max:.1f}")
            print(f"   ✓ Sweet spot: {rsi_25pct:.1f} - {rsi_75pct:.1f} (your 25th-75th percentile)")
            print(f"   ✓ Best performance RSI: {best_exit['entry_rsi14_w_mean']:.1f} ± {best_exit['entry_rsi14_w_std']:.1f}")
        
        # 2. VWAP Logic
        if 'entry_below_vwap_pct' in patterns_full.columns:
            below_vwap_avg = type_patterns['entry_below_vwap_pct'].mean()
            vwap_dist_mean = type_patterns['entry_vwap_distance_mean'].mean()
            
            print(f"\n2️⃣ VWAP CONDITIONS:")
            if below_vwap_avg > 60:
                print(f"   ✓ Price should be BELOW VWAP ({below_vwap_avg:.0f}% of your {trade_type}s)")
            elif below_vwap_avg < 40:
                print(f"   ✓ Price should be ABOVE VWAP ({100-below_vwap_avg:.0f}% of your {trade_type}s)")
            else:
                print(f"   ✓ Price can be either side of VWAP (no strong bias)")
            
            if abs(vwap_dist_mean) > 0.01:
                print(f"   ✓ Typical distance from VWAP: {vwap_dist_mean:+.3f}%")
        
        # 3. Volume Logic
        if 'entry_rvol20_mean' in patterns_full.columns:
            rvol_mean = type_patterns['entry_rvol20_mean'].mean()
            rvol_min = type_patterns['entry_rvol20_min'].min()
            rvol_max = type_patterns['entry_rvol20_max'].max()
            
            print(f"\n3️⃣ VOLUME CONDITIONS:")
            print(f"   ✓ RVOL typically around {rvol_mean:.2f}x")
            print(f"   ✓ Acceptable range: {rvol_min:.2f}x to {rvol_max:.2f}x")
            if rvol_mean > 2:
                print(f"   ✓ Higher volume preferred (avg: {rvol_mean:.2f}x)")
            elif rvol_mean < 1:
                print(f"   ✓ Lower volume preferred (avg: {rvol_mean:.2f}x)")
        
        # 4. StochRSI Logic
        if 'entry_stochrsi_k_mean' in patterns_full.columns:
            stoch_mean = type_patterns['entry_stochrsi_k_mean'].mean()
            stoch_min = type_patterns['entry_stochrsi_k_min'].min()
            stoch_max = type_patterns['entry_stochrsi_k_max'].max()
            
            if not np.isnan(stoch_mean):
                print(f"\n4️⃣ STOCHRSI CONDITIONS:")
                print(f"   ✓ StochRSI K range: {stoch_min:.1f} to {stoch_max:.1f}")
                print(f"   ✓ Typical value: {stoch_mean:.1f}")
        
        # 5. EMA Logic
        if 'entry_below_ema9_pct' in patterns_full.columns:
            print(f"\n5️⃣ EMA CONDITIONS:")
            
            for ema in ['ema9', 'ema20', 'ema50']:
                col_name = f'entry_below_{ema}_pct'
                if col_name in type_patterns.columns:
                    below_ema = type_patterns[col_name].mean()
                    if below_ema > 70:
                        print(f"   ✓ Price typically BELOW {ema.upper()} ({below_ema:.0f}% of trades)")
                    elif below_ema < 30:
                        print(f"   ✓ Price typically ABOVE {ema.upper()} ({100-below_ema:.0f}% of trades)")
        
        # 6. ATR/Volatility Logic
        if 'entry_atr14_w_mean' in patterns_full.columns:
            atr_mean = type_patterns['entry_atr14_w_mean'].mean()
            atr_max = type_patterns['entry_atr14_w_max'].max()
            
            print(f"\n6️⃣ VOLATILITY CONDITIONS:")
            print(f"   ✓ Typical ATR: {atr_mean:.3f}")
            print(f"   ✓ Max ATR traded: {atr_max:.3f}")
            print(f"   ⚠️ Avoid if ATR > {atr_max:.3f}")
        
        # 7. Time of Day Logic
        if 'entry_hour_mode' in patterns_full.columns:
            hour_mode = int(type_patterns['entry_hour_mode'].mode()[0]) if len(type_patterns['entry_hour_mode'].mode()) > 0 else type_patterns['entry_hour_mean'].mean()
            
            print(f"\n7️⃣ TIME CONDITIONS:")
            print(f"   ✓ Most active hour: {int(hour_mode):02d}:00")
            print(f"   ✓ Focus on morning hours for best setups")
        
        # 8. Momentum Logic (from indicator changes)
        if 'rsi14_w_change_mean' in patterns_full.columns:
            rsi_change = type_patterns['rsi14_w_change_mean'].mean()
            
            print(f"\n8️⃣ MOMENTUM CONDITIONS:")
            if trade_type == 'CALL' and rsi_change > 0:
                print(f"   ✓ Look for positive RSI momentum (avg change: +{rsi_change:.1f})")
            elif trade_type == 'PUT' and rsi_change < 0:
                print(f"   ✓ Look for negative RSI momentum (avg change: {rsi_change:.1f})")
        
        # Exit Strategy
        print(f"\n🎯 {trade_type} EXIT STRATEGY:")
        for exit_type in ['EXIT', 'STOP_LOSS', 'RUNNER']:
            pattern_key = f"{trade_type}_{exit_type}"
            if pattern_key in patterns_full.index:
                exit_data = patterns_full.loc[pattern_key]
                print(f"\n   {exit_type}:")
                print(f"   • Duration: {exit_data['avg_duration']:.1f} minutes")
                print(f"   • Avg Return: {exit_data['avg_return']:.3f}%")
                print(f"   • Win Rate: {exit_data['profitable_pct']:.1f}%")

print("\n" + "="*60)
print("END OF LOGICAL CHECKLIST")
print("="*60)

## 4. Current Indicator Analysis

In [ ]:
print("\n" + "="*60)
print("INDICATOR STATUS vs YOUR PATTERNS")
print("="*60)

# RSI Analysis
current_rsi = current['RSI14_W']
call_rsi_min, call_rsi_max = YOUR_PATTERNS['CALL']['rsi_range']
put_rsi_min, put_rsi_max = YOUR_PATTERNS['PUT']['rsi_range']

print(f"\nRSI Analysis:")
print(f"  Current RSI: {current_rsi:.1f}")

if call_rsi_min <= current_rsi <= call_rsi_max:
    print(f"  ✅ In CALL range ({call_rsi_min:.0f}-{call_rsi_max:.0f})")
else:
    print(f"  ❌ Outside CALL range ({call_rsi_min:.0f}-{call_rsi_max:.0f})")

if put_rsi_min <= current_rsi <= put_rsi_max:
    print(f"  ✅ In PUT range ({put_rsi_min:.0f}-{put_rsi_max:.0f})")
else:
    print(f"  ❌ Outside PUT range ({put_rsi_min:.0f}-{put_rsi_max:.0f})")

# StochRSI Analysis
print(f"\nStochRSI Analysis:")
if pd.notna(current['StochRSI_K']):
    print(f"  StochRSI K: {current['StochRSI_K']:.1f}")
    print(f"  StochRSI D: {current['StochRSI_D']:.1f}")
else:
    print("  StochRSI: No data (null values)")

# Volume Analysis
current_rvol = current['RVOL20']
print(f"\nVolume Analysis:")
print(f"  Current RVOL: {current_rvol:.2f}x")
print(f"  CALL typical RVOL: {YOUR_PATTERNS['CALL']['rvol_avg']:.1f}x")
print(f"  PUT typical RVOL: {YOUR_PATTERNS['PUT']['rvol_avg']:.1f}x")

# ATR for volatility
print(f"\nVolatility:")
print(f"  Current ATR: {current['ATR14_W']:.3f}")

# Calculate average ATR from enriched trades for comparison
enriched_atr_avg = enriched_df['Entry_ATR14_W'].mean()
print(f"  Your typical ATR: {enriched_atr_avg:.3f}")

## 5. Trade Setup Detection

In [ ]:
print("\n" + "="*60)
print("TRADE SETUP ANALYSIS")
print("="*60)

# Check recent price movement
if len(today_data) >= 5:
    recent_5min = today_data.tail(5)
    price_change_5m = (current['Last'] - recent_5min.iloc[0]['Last']) / recent_5min.iloc[0]['Last'] * 100
    
    print(f"\n5-Minute Price Change: {price_change_5m:+.3f}%")
    
    # CALL Setup Check based on actual patterns
    call_score = 0
    call_conditions = []
    
    if call_rsi_min <= current_rsi <= call_rsi_max:
        call_score += 1
        call_conditions.append(f"✅ RSI in your range ({call_rsi_min:.0f}-{call_rsi_max:.0f})")
    else:
        call_conditions.append(f"❌ RSI out of your range (current: {current_rsi:.1f})")
    
    if price_change_5m > 0:
        call_score += 1
        call_conditions.append("✅ Upward momentum")
    else:
        call_conditions.append("❌ No upward momentum")
    
    if not above_vwap:  # Your CALLs work below VWAP
        call_score += 0.5
        call_conditions.append(f"✅ Below VWAP (matches {YOUR_PATTERNS['CALL']['below_vwap_pct']*100:.0f}% of your CALLs)")
    else:
        call_conditions.append(f"⚠️ Above VWAP (only {(1-YOUR_PATTERNS['CALL']['below_vwap_pct'])*100:.0f}% of your CALLs)")
    
    if current_rvol > 2:
        call_score += 0.5
        call_conditions.append(f"✅ Higher volume (your CALL avg: {YOUR_PATTERNS['CALL']['rvol_avg']:.1f}x)")
    
    print(f"\nCALL Setup Analysis (Score: {call_score}/3):")
    for condition in call_conditions:
        print(f"  {condition}")
    
    # PUT Setup Check based on actual patterns
    put_score = 0
    put_conditions = []
    
    if put_rsi_min <= current_rsi <= put_rsi_max:
        put_score += 1
        put_conditions.append(f"✅ RSI in your range ({put_rsi_min:.0f}-{put_rsi_max:.0f})")
    else:
        put_conditions.append(f"❌ RSI out of your range (current: {current_rsi:.1f})")
    
    if price_change_5m < 0:
        put_score += 1
        put_conditions.append("✅ Downward momentum")
    else:
        put_conditions.append("❌ No downward momentum")
    
    if above_vwap:  # Your PUTs work above VWAP
        put_score += 0.5
        put_conditions.append(f"✅ Above VWAP (matches {YOUR_PATTERNS['PUT']['above_vwap_pct']*100:.0f}% of your PUTs)")
    else:
        put_conditions.append(f"⚠️ Below VWAP (only {(1-YOUR_PATTERNS['PUT']['above_vwap_pct'])*100:.0f}% of your PUTs)")
    
    if current_rvol < 1.5:
        put_score += 0.5
        put_conditions.append(f"✅ Lower volume (your PUT avg: {YOUR_PATTERNS['PUT']['rvol_avg']:.1f}x)")
    
    print(f"\nPUT Setup Analysis (Score: {put_score}/3):")
    for condition in put_conditions:
        print(f"  {condition}")
else:
    print("\nNot enough data points for 5-minute analysis")
    call_score = 0
    put_score = 0

## 6. Recent Similar Trades

In [ ]:
# Load similar trades and show recent ones
similar_df = pd.read_csv('data/similar_trades_pipeline.csv')
similar_df['Entry_Time'] = pd.to_datetime(similar_df['Entry_Time'])

# Get trades from last few days matching current conditions
print("\n" + "="*60)
print("RECENT SIMILAR PROFITABLE TRADES")
print("="*60)

# Filter for trades with similar RSI conditions
rsi_tolerance = 5
similar_conditions = similar_df[
    (abs(similar_df['Entry_RSI'] - current_rsi) <= rsi_tolerance)
].copy()

if len(similar_conditions) > 0:
    # Sort by Expected Return to show best opportunities
    similar_conditions = similar_conditions.sort_values('Expected_Return', ascending=False)
    
    print(f"\nFound {len(similar_conditions)} trades with RSI {current_rsi-rsi_tolerance:.0f}-{current_rsi+rsi_tolerance:.0f}")
    
    # Show top similar trades by type
    for trade_type in ['CALL', 'PUT']:
        type_trades = similar_conditions[similar_conditions['Trade_Type'] == trade_type]
        if len(type_trades) > 0:
            print(f"\nTop {trade_type} opportunities:")
            display_cols = ['Entry_Time', 'Entry_RSI', 'Exit_Duration', 'Expected_Return']
            print(type_trades.head(5)[display_cols].to_string(index=False))
            
            # Calculate average metrics
            avg_duration = type_trades['Exit_Duration'].mean()
            avg_return = type_trades['Expected_Return'].mean()
            print(f"\n  Average {trade_type} duration: {avg_duration:.0f} minutes")
            print(f"  Average {trade_type} return: {avg_return:.3f}%")
else:
    print(f"\nNo similar trades found with RSI near {current_rsi:.1f}")

## 7. Trading Recommendations

In [ ]:
print("\n" + "="*60)
print("TRADING RECOMMENDATIONS")
print("="*60)

# Decision logic based on scores
if call_score >= 2.5:
    print("\n🟢 CALL SETUP DETECTED")
    print(f"  - Score: {call_score}/3")
    print(f"  - Your typical hold time: {YOUR_PATTERNS['CALL']['duration_avg']:.0f} minutes")
    print(f"  - Expected return: {YOUR_PATTERNS['CALL']['avg_return']:.2f}% stock move")
    print(f"  - Historical win rate: {YOUR_PATTERNS['CALL']['win_rate']*100:.0f}%")
    
    # Get specific durations from enriched data
    call_exits = enriched_df[(enriched_df['Trade_Type'] == 'CALL') & (enriched_df['Exit_Type'] == 'EXIT')]
    call_stops = enriched_df[(enriched_df['Trade_Type'] == 'CALL') & (enriched_df['Exit_Type'] == 'STOP_LOSS')]
    call_runners = enriched_df[(enriched_df['Trade_Type'] == 'CALL') & (enriched_df['Exit_Type'] == 'RUNNER')]
    
    if len(call_exits) > 0:
        print(f"\n  Exit targets from your data:")
        print(f"    - Quick exit: {call_exits['Duration'].mean():.0f} min")
    if len(call_stops) > 0:
        print(f"    - Stop loss: {call_stops['Duration'].mean():.0f} min")
    if len(call_runners) > 0:
        print(f"    - Runner: {call_runners['Duration'].mean():.0f} min")
        
elif put_score >= 2.5:
    print("\n🔴 PUT SETUP DETECTED")
    print(f"  - Score: {put_score}/3")
    print(f"  - Your typical hold time: {YOUR_PATTERNS['PUT']['duration_avg']:.0f} minutes")
    print(f"  - Expected return: {YOUR_PATTERNS['PUT']['avg_return']:.2f}% stock move")
    print(f"  - Historical win rate: {YOUR_PATTERNS['PUT']['win_rate']*100:.0f}%")
    
    # Get specific durations from enriched data
    put_exits = enriched_df[(enriched_df['Trade_Type'] == 'PUT') & (enriched_df['Exit_Type'] == 'EXIT')]
    put_stops = enriched_df[(enriched_df['Trade_Type'] == 'PUT') & (enriched_df['Exit_Type'] == 'STOP_LOSS')]
    put_runners = enriched_df[(enriched_df['Trade_Type'] == 'PUT') & (enriched_df['Exit_Type'] == 'RUNNER')]
    
    if len(put_exits) > 0:
        print(f"\n  Exit targets from your data:")
        print(f"    - Quick exit: {put_exits['Duration'].mean():.0f} min")
    if len(put_stops) > 0:
        print(f"    - Stop loss: {put_stops['Duration'].mean():.0f} min")
    if len(put_runners) > 0:
        print(f"    - Runner: {put_runners['Duration'].mean():.0f} min (BEST RETURNS!)")
else:
    print("\n⚪ NO CLEAR SETUP")
    print(f"  - CALL score: {call_score}/3 (need 2.5+)")
    print(f"  - PUT score: {put_score}/3 (need 2.5+)")
    print("  - Wait for better conditions")
    print("  - Your setups require specific RSI ranges and momentum")

# Key reminders based on actual data
print("\n" + "="*60)
print("REMINDERS FROM YOUR ACTUAL TRADING DATA")
print("="*60)

# Get actual extremes from data
call_data = enriched_df[enriched_df['Trade_Type'] == 'CALL']
put_data = enriched_df[enriched_df['Trade_Type'] == 'PUT']

if len(call_data) > 0:
    print(f"✓ Your CALL trades: RSI {call_data['Entry_RSI14_W'].min():.0f}-{call_data['Entry_RSI14_W'].max():.0f}")
    print(f"✓ {(call_data['Entry_Last'] < call_data['Entry_VWAP']).sum()/len(call_data)*100:.0f}% of your CALLs are BELOW VWAP")

if len(put_data) > 0:
    print(f"✓ Your PUT trades: RSI {put_data['Entry_RSI14_W'].min():.0f}-{put_data['Entry_RSI14_W'].max():.0f}")
    print(f"✓ {(put_data['Entry_Last'] > put_data['Entry_VWAP']).sum()/len(put_data)*100:.0f}% of your PUTs are ABOVE VWAP")

print(f"✓ Average returns: CALL {YOUR_PATTERNS['CALL']['avg_return']:.2f}%, PUT {YOUR_PATTERNS['PUT']['avg_return']:.2f}%")

## 8. Market Internals Check

In [ ]:
# Additional market context
print("\n" + "="*60)
print("ADDITIONAL MARKET CONTEXT")
print("="*60)

# Check for unusual conditions based on your data
high_rvol_threshold = enriched_df['Entry_RVOL20'].quantile(0.95)
high_atr_threshold = enriched_df['Entry_ATR14_W'].quantile(0.95)

if current_rvol > high_rvol_threshold:
    print(f"⚠️ WARNING: Extremely high volume ({current_rvol:.1f}x) - above your 95th percentile ({high_rvol_threshold:.1f}x)")
    
if current['ATR14_W'] > high_atr_threshold:
    print(f"⚠️ WARNING: High volatility (ATR: {current['ATR14_W']:.3f}) - above your 95th percentile ({high_atr_threshold:.3f})")
    
# Check if RSI is outside your trading ranges
if current_rsi < min(call_rsi_min, put_rsi_min):
    print(f"⚠️ WARNING: RSI {current_rsi:.1f} is below your lowest traded level ({min(call_rsi_min, put_rsi_min):.0f})")
elif current_rsi > max(call_rsi_max, put_rsi_max):
    print(f"⚠️ WARNING: RSI {current_rsi:.1f} is above your highest traded level ({max(call_rsi_max, put_rsi_max):.0f})")

# Time of day analysis
current_hour = current['Time'].hour
current_minute = current['Time'].minute

print(f"\n📍 Time Analysis: {current_hour:02d}:{current_minute:02d}")

# Analyze your trading patterns by time
enriched_df['Entry_Hour'] = pd.to_datetime(enriched_df['Entry_Time']).dt.hour
hour_counts = enriched_df['Entry_Hour'].value_counts().sort_index()

if current_hour in hour_counts.index:
    trades_this_hour = hour_counts[current_hour]
    print(f"  - You've made {trades_this_hour} trades during the {current_hour}:00 hour")
else:
    print(f"  - You have no historical trades during the {current_hour}:00 hour")

# Show your most active trading hours
print("\n  Your most active trading hours:")
for hour, count in hour_counts.head(3).items():
    print(f"    - {hour:02d}:00: {count} trades")

print("\n" + "="*60)
print("Analysis Complete - Trade YOUR Patterns!")
print("="*60)